# Medical Model Inversion Privacy Assessment

Complete the TODOs in `src/medical_inversion_utils.py`, then run this notebook to generate your privacy assessment deliverables.

In [ ]:
from pathlib import Path
import os, sys
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))
os.environ.setdefault("ART_DATA_PATH", str(ROOT / "art_data"))

from medical_inversion_utils import (
    art_status, evaluate_model_outputs, inversion_metrics, output_configurations,
    plot_leakage_scores, plot_reconstruction_comparison, prepare_medical_dataset,
    representative_samples, run_inversion_attack, seed_everything,
    train_or_load_model, write_csv, write_json, write_privacy_report,
)

seed_everything(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
device, art_status()

In [ ]:
dataset = prepare_medical_dataset(ROOT / "data" / "generated")
model = train_or_load_model(ROOT / "models" / "brain_tumor_screening_cnn.pt", dataset, device=device)
baseline_metrics, confidence_rows, probs = evaluate_model_outputs(
    model, dataset.val_images, dataset.val_labels, dataset.class_names, device=device
)
write_csv(confidence_rows, ROOT / "results" / "baseline_confidence_outputs.csv")
baseline_metrics

## TODO

Implement `output_configurations`, `run_inversion_attack`, `inversion_metrics`, and `write_privacy_report` in `src/medical_inversion_utils.py` before running the remaining cells.

In [ ]:
target_classes = list(range(len(dataset.class_names)))
representatives = representative_samples(dataset.val_images, dataset.val_labels, target_classes)
reconstruction_sets = {}
metric_rows = []

for config in output_configurations():
    reconstructions = run_inversion_attack(
        model,
        target_classes,
        output_config=config,
        device=device,
        image_size=int(dataset.train_images.shape[-1]),
    )
    reconstruction_sets[config.name] = reconstructions
    metric_rows.extend(inversion_metrics(config.name, reconstructions, representatives, dataset.class_names))

write_csv(metric_rows, ROOT / "results" / "model_inversion_privacy_metrics.csv")
metric_rows[:3]

In [ ]:
comparison_path = plot_reconstruction_comparison(
    reconstruction_sets, representatives, dataset.class_names, ROOT / "results" / "reconstructed_medical_features.png"
)
chart_path = plot_leakage_scores(metric_rows, ROOT / "results" / "privacy_leakage_by_output_config.png")
summary_path = write_json(
    {"baseline_model": baseline_metrics, "art_status": art_status(), "attack_metrics": metric_rows},
    ROOT / "results" / "privacy_assessment_summary.json",
)
report_path = write_privacy_report(
    ROOT / "results" / "privacy_assessment_report.md",
    baseline_metrics,
    metric_rows,
    chart_path,
    comparison_path,
)
comparison_path, chart_path, summary_path, report_path